# Recreating a `fullosc` (numu -> nue appearance) file, with systematic error bands

**Goal.** Build an approximate "full oscillation" (numu -> nue appearance) sample by pairing
numu and nue events, and make prediction plots with a systematic error band plus a
fractional-uncertainty breakdown from **XS (GENIE)**, **flux**, and **DetVar**, reusing this
project's existing systematics + plotting functions.

**Run period.** We use **run 4d** for both the numu (nu_overlay) and nue (intrinsic nue)
samples, because the DetVar samples exist for run 4d nu_overlay. (The official `numu2nue`
reference on the gpvm is run 3; we assume run 3 ~ run 4d for the shape comparison.)

**Method.**
1. numu donors = preselected run-4d `nu_overlay` events (carry the numu flux normalization).
2. Reweight each by the nue/numu total cross section ratio `sigma_nueCC(E)/sigma_numu_total(E)`
   at its true energy -- a **PLACEHOLDER** here (the repo has no absolute sigma(E) spline),
   to be swapped for a real GENIE spline.
3. Pair each numu donor to the nearest-true-energy preselected run-4d `nue_overlay` (nueCC)
   template, and take that nue template's reco + truth.
4. Systematics for the recreated event:
   - **flux** multisim: from the **numu donor** (`flux_all`).
   - **XS / GENIE** multisim (`All_UBGenie` + unisim knobs) and **reinteraction** (`reint_all`):
     from the **nue template** (the recreated event is a nueCC interaction).
   - **DetVar**: from the run-4d `nu_overlay` detector-variation samples (the only DetVar that
     exists) -- applied as a fractional uncertainty vs reco energy (see caveat in that section).

**Why source from the intermediate parquets.** The project's systematics functions
(`create_rw_frac_cov_matrices`, `create_detvar_frac_cov_matrices`) join per-event multisim
weights from `presel_weights_df.parquet` / `detvar_presel_df_train_vars.parquet` on
`(filename, run, subrun, event)`. So the main pipeline reads the **preselected** parquets
produced by `create_df.py` / `create_rw_syst_df.py` / `create_detvar_df.py` (these are derived
from the raw run-4d checkout files). A raw-uproot reader is kept at the end only for the
reference-file comparison.

## 0. Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.file_locations import intermediate_files_location, data_files_location
from src.systematics import (
    create_rw_frac_cov_matrices,
    create_detvar_frac_cov_matrices,
    get_pred_stat_cov,
)
from src.plot_helpers import make_sys_frac_error_plot

RUN_PERIOD = "4d"

# reco-energy binning for all histograms / covariances
BINS = np.linspace(0, 2500, 26)          # 100 MeV bins, 0-2500 MeV
BIN_CENTERS = 0.5 * (BINS[:-1] + BINS[1:])
RECO_VAR = "wc_kine_reco_Enu"            # column in the parquets (MeV)

# net-weight config to use (real POT). open_data maps run 4d -> run-4b open-data POT.
NET_WEIGHT_VAR = "wc_net_weight_open_data"

# selection flags
REQUIRE_FV = True                        # true vertex inside FV for both donors and templates

# optional subsample of paired events for the (heavier) systematic covariances.
# None = use all paired events. Weights are rescaled so the normalization is preserved.
SYST_MAX_EVENTS = None

# DetVar: bootstrapping is the analysis default (statistical treatment of CV-var diff) but is
# slow (5000 rounds x 7 vartypes). False is much faster and fine for a first look.
USE_DETVAR_BOOTSTRAPPING = False

## 1. Load the preselected run-4d prediction, weights, and DetVar dataframes

In [ ]:
PRED_COLS = ["filetype", "filename", "run", "subrun", "event", "detailed_run_period",
             RECO_VAR, "wc_truth_nuEnergy", "wc_truth_nuPdg", "wc_truth_isCC",
             "wc_truth_vtxInside", "wc_weight_cv", "wc_weight_spline", NET_WEIGHT_VAR]

pred_lf = (pl.scan_parquet(f"{intermediate_files_location}/presel_df_train_vars.parquet")
           .filter(pl.col("detailed_run_period") == RUN_PERIOD)
           .select(PRED_COLS))

# numu donors: run-4d nu_overlay, |pdg|==14 (CC+NC), FV
numu_df = pred_lf.filter(
    (pl.col("filetype") == "nu_overlay") & (pl.col("wc_truth_nuPdg").abs() == 14)
    & (pl.col("wc_truth_vtxInside") if REQUIRE_FV else pl.lit(True))
).collect()

# nue templates: run-4d nue_overlay, nueCC (|pdg|==12 & isCC), FV
nue_df = pred_lf.filter(
    (pl.col("filetype") == "nue_overlay") & (pl.col("wc_truth_nuPdg").abs() == 12)
    & pl.col("wc_truth_isCC")
    & (pl.col("wc_truth_vtxInside") if REQUIRE_FV else pl.lit(True))
).collect()

print(f"numu donors (run {RUN_PERIOD}, nu_overlay, |pdg|=14, FV):  {numu_df.height:,}")
print(f"nue templates (run {RUN_PERIOD}, nue_overlay, nueCC, FV):  {nue_df.height:,}")

# preloaded weights (all universe columns) for run-4d nu+nue -> used by the systematics calls
UNIV_COLS = ["All_UBGenie", "flux_all", "reint_all",
             "AxFFCCQEshape_UBGenie", "DecayAngMEC_UBGenie", "NormCCCOH_UBGenie",
             "NormNCCOH_UBGenie", "RPA_CCQE_UBGenie", "ThetaDelta2NRad_UBGenie",
             "Theta_Delta2Npi_UBGenie", "VecFFCCQEshape_UBGenie", "XSecShape_CCMEC_UBGenie",
             "xsr_scc_Fa3_SCC", "xsr_scc_Fv3_SCC"]
weights_run4d = (pl.scan_parquet(f"{intermediate_files_location}/presel_weights_df.parquet")
                 .filter((pl.col("detailed_run_period") == RUN_PERIOD)
                         & (pl.col("filetype").is_in(["nu_overlay", "nue_overlay"])))
                 .select(["filename", "run", "subrun", "event"] + UNIV_COLS)
                 .collect(engine="streaming"))
print(f"weights rows (run {RUN_PERIOD} nu+nue): {weights_run4d.height:,}")

# DetVar dataframe (run-4d nu_overlay CV + 7 variations)
detvar_df = (pl.scan_parquet(f"{intermediate_files_location}/detvar_presel_df_train_vars.parquet")
             .filter(pl.col("detailed_run_period") == RUN_PERIOD)
             .select(["filetype", "run", "subrun", "event", "vartype", "wc_net_weight", RECO_VAR])
             .collect())
print("DetVar vartypes:", detvar_df["vartype"].unique().sort().to_list())

## 2. Cross-section ratio  sigma_nueCC(E) / sigma_numu_total(E)  -- PLACEHOLDER

Same placeholder as before: **not** a real GENIE cross section. Replace with a lookup of a real
GENIE total-cross-section spline (sigma vs E on Ar40 for nu_e CC and nu_mu total).

In [ ]:
def xsec_ratio_nueCC_over_numuTotal(E_MeV):
    """PLACEHOLDER sigma_nueCC(E)/sigma_numu_total(E). REPLACE WITH A REAL GENIE SPLINE.
    ratio ~ [1 + 0.20*exp(-E/300 MeV)] * 0.75  (mild low-E nueCC boost x CC fraction)."""
    E = np.asarray(E_MeV, dtype=np.float64)
    return (1.0 + 0.20 * np.exp(-E / 300.0)) * 0.75

_E = np.linspace(50, 3000, 300)
plt.figure(figsize=(6, 3.5))
plt.plot(_E, xsec_ratio_nueCC_over_numuTotal(_E))
plt.xlabel("true neutrino energy [MeV]"); plt.ylabel(r"$\sigma_{\nu_eCC}/\sigma_{\nu_\mu tot}$")
plt.title("PLACEHOLDER cross-section ratio"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3. Pair each numu donor to the nearest-true-energy nue template

In [ ]:
numu_E = numu_df["wc_truth_nuEnergy"].to_numpy().astype(np.float64)
nue_E = nue_df["wc_truth_nuEnergy"].to_numpy().astype(np.float64)

order = np.argsort(nue_E)
nue_E_sorted = nue_E[order]

def nearest_nue_index(E):
    pos = np.clip(np.searchsorted(nue_E_sorted, E), 1, len(nue_E_sorted) - 1)
    take_left = (E - nue_E_sorted[pos - 1]) <= (nue_E_sorted[pos] - E)
    return order[np.where(take_left, pos - 1, pos)]

match_idx = nearest_nue_index(numu_E)
dE = numu_E - nue_E[match_idx]
print(f"paired {len(numu_E):,} numu donors to nue templates")
print("  |dE| [MeV]: median %.2f  95pct %.2f  max %.2f"
      % (np.median(np.abs(dE)), np.percentile(np.abs(dE), 95), np.abs(dE).max()))
print(f"  unique nue templates used: {len(np.unique(match_idx)):,} of {nue_df.height:,}")

plt.figure(figsize=(6, 3.5))
plt.hist(dE, bins=100, range=(-50, 50))
plt.xlabel("E(numu donor) - E(nue template) [MeV]"); plt.ylabel("events")
plt.title("energy-pairing residual"); plt.tight_layout(); plt.show()

## 4. Assemble the recreated fullosc prediction (CV level)

Each row = one numu donor (unique) paired to a nue template. Reco/truth come from the nue
template; the weight = numu donor net weight x XS ratio(E_numu). We keep both event identities
(and both CV weights) so the systematics calls can pull flux weights from the numu donor and
XS weights from the nue template.

In [ ]:
nue_m = nue_df[match_idx]     # matched nue templates, aligned row-for-row with numu_df

xs_ratio = xsec_ratio_nueCC_over_numuTotal(numu_E)
fullosc_net_weight = numu_df[NET_WEIGHT_VAR].to_numpy().astype(np.float64) * xs_ratio

fullosc = pl.DataFrame({
    # reco + truth from the nue template
    "reco_Enu": nue_m[RECO_VAR].to_numpy(),
    "truth_nuEnergy": nue_m["wc_truth_nuEnergy"].to_numpy(),
    # weight
    "fullosc_net_weight": fullosc_net_weight,
    "xs_ratio": xs_ratio,
    # numu donor identity + CV weights (for flux systematics)
    "numu_filename": numu_df["filename"].to_numpy(),
    "numu_run": numu_df["run"].to_numpy(),
    "numu_subrun": numu_df["subrun"].to_numpy(),
    "numu_event": numu_df["event"].to_numpy(),
    "numu_weight_cv": numu_df["wc_weight_cv"].to_numpy(),
    "numu_weight_spline": numu_df["wc_weight_spline"].to_numpy(),
    # nue template identity + CV weights (for XS systematics)
    "nue_filename": nue_m["filename"].to_numpy(),
    "nue_run": nue_m["run"].to_numpy(),
    "nue_subrun": nue_m["subrun"].to_numpy(),
    "nue_event": nue_m["event"].to_numpy(),
    "nue_weight_cv": nue_m["wc_weight_cv"].to_numpy(),
    "nue_weight_spline": nue_m["wc_weight_spline"].to_numpy(),
})

# keep only reconstructed events (preselected -> kine_reco_Enu > 0)
fullosc = fullosc.filter(pl.col("reco_Enu") > 0)
print(f"recreated fullosc events (reco > 0): {fullosc.height:,}")
print(f"sum of weights @ run-{RUN_PERIOD} open-data POT: {fullosc['fullosc_net_weight'].sum():.2f}")

## 5. CV sanity plots: true and reco neutrino energy

In [ ]:
w = fullosc["fullosc_net_weight"].to_numpy()
reco = fullosc["reco_Enu"].to_numpy()
true = fullosc["truth_nuEnergy"].to_numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(true, bins=BINS, weights=w, histtype="step")
ax[0].set_xlabel("true neutrino energy [MeV]"); ax[0].set_title("recreated true Enu (nueCC template)")
ax[1].hist(reco, bins=BINS, weights=w, histtype="step")
ax[1].set_xlabel("reco neutrino energy [MeV]"); ax[1].set_title("recreated reco Enu")
for a in ax: a.set_ylabel(f"events / bin @ open-data POT")
plt.tight_layout(); plt.show()

## 6. Systematic covariance matrices (XS, flux, DetVar)

We reuse `create_rw_frac_cov_matrices` twice on the **same paired events** (identical reco
values and net weight), differing only in which event identity + CV weight is used:

- keyed by the **numu-donor** identity -> keep the **flux** fractional covariance.
- keyed by the **nue-template** identity (+ nue CV weight) -> keep **All_UBGenie**, the GENIE
  unisim knobs, and **reinteraction**.

DetVar comes from `create_detvar_frac_cov_matrices` on the run-4d nu_overlay DetVar samples.

In [ ]:
# optional subsample (rescale weights so the normalization is preserved)
n_all = fullosc.height
if SYST_MAX_EVENTS is not None and n_all > SYST_MAX_EVENTS:
    fos = fullosc.sample(n=SYST_MAX_EVENTS, seed=0)
    scale = n_all / SYST_MAX_EVENTS
    print(f"using {SYST_MAX_EVENTS:,}/{n_all:,} paired events for systematics (weights x{scale:.2f})")
else:
    fos = fullosc
    scale = 1.0
    print(f"using all {n_all:,} paired events for systematics")

reco_s = fos["reco_Enu"].to_numpy()
w_s = fos["fullosc_net_weight"].to_numpy() * scale

# mc_pred_df keyed by the NUMU-donor identity -> flux
mc_numu = fos.select([
    pl.col("numu_filename").alias("filename"),
    pl.col("numu_run").alias("run"), pl.col("numu_subrun").alias("subrun"),
    pl.col("numu_event").alias("event"),
    (pl.col("fullosc_net_weight") * scale).alias("fullosc_net_weight"),
    pl.col("numu_weight_cv").alias("wc_weight_cv"),
    pl.col("numu_weight_spline").alias("wc_weight_spline"),
    pl.col("reco_Enu").alias("reco_Enu"),
])
# mc_pred_df keyed by the NUE-template identity (+ nue CV weight) -> All_UBGenie, unisims, reint
mc_nue = fos.select([
    pl.col("nue_filename").alias("filename"),
    pl.col("nue_run").alias("run"), pl.col("nue_subrun").alias("subrun"),
    pl.col("nue_event").alias("event"),
    (pl.col("fullosc_net_weight") * scale).alias("fullosc_net_weight"),
    pl.col("nue_weight_cv").alias("wc_weight_cv"),
    pl.col("nue_weight_spline").alias("wc_weight_spline"),
    pl.col("reco_Enu").alias("reco_Enu"),
])

rw_numu = create_rw_frac_cov_matrices(mc_numu, "reco_Enu", BINS, weights_df=weights_run4d,
                                      net_weight_var="fullosc_net_weight")
rw_nue = create_rw_frac_cov_matrices(mc_nue, "reco_Enu", BINS, weights_df=weights_run4d,
                                     net_weight_var="fullosc_net_weight")

# merge: flux from numu; All_UBGenie + unisims + reinteraction from nue
GENIE_KEYS = ["All_UBGenie", "AxFFCCQEshape_UBGenie", "DecayAngMEC_UBGenie", "NormCCCOH_UBGenie",
              "NormNCCOH_UBGenie", "RPA_CCQE_UBGenie", "ThetaDelta2NRad_UBGenie",
              "Theta_Delta2Npi_UBGenie", "VecFFCCQEshape_UBGenie", "XSecShape_CCMEC_UBGenie",
              "xsr_scc_Fa3_SCC", "xsr_scc_Fv3_SCC"]
rw_sys_frac_cov_dic = {"flux": rw_numu["flux"], "reinteraction": rw_nue["reinteraction"]}
for k in GENIE_KEYS:
    rw_sys_frac_cov_dic[k] = rw_nue[k]

# DetVar fractional covariances (from run-4d nu_overlay detector variations)
detvar_sys_frac_cov_dic = create_detvar_frac_cov_matrices(
    detvar_df, RECO_VAR, BINS, use_detvar_bootstrapping=USE_DETVAR_BOOTSTRAPPING)

print("rw keys:", list(rw_sys_frac_cov_dic.keys()))
print("detvar keys:", list(detvar_sys_frac_cov_dic.keys()))

## 7. Prediction with total systematic error band

The predicted reco-energy spectrum (all paired events) with a 1-sigma band from
XS + flux + reinteraction + DetVar (+ MC stat).

In [ ]:
# full-statistics prediction counts (all paired events)
pred_counts = np.histogram(reco, weights=w, bins=BINS)[0]
pred_counts_safe = np.maximum(pred_counts, 1e-9)

# combine fractional covariances (all are fractional relative to the prediction)
nb = len(BINS) - 1
tot_frac = np.zeros((nb, nb))
for m in rw_sys_frac_cov_dic.values():
    tot_frac += m
for m in detvar_sys_frac_cov_dic.values():
    tot_frac += m

# MC-stat fractional covariance
pred_stat_cov = get_pred_stat_cov(reco, w, BINS)
denom = np.outer(pred_counts_safe, pred_counts_safe)
pred_stat_frac = np.nan_to_num(pred_stat_cov / denom, nan=0, posinf=0, neginf=0)

tot_frac_with_stat = tot_frac + pred_stat_frac
band = pred_counts * np.sqrt(np.clip(np.diag(tot_frac_with_stat), 0, None))

plt.figure(figsize=(8, 5))
plt.hist(BIN_CENTERS, bins=BINS, weights=pred_counts, histtype="step", color="C0", lw=2,
         label="recreated fullosc prediction")
plt.bar(BIN_CENTERS, height=2 * band, width=np.diff(BINS), bottom=pred_counts - band,
        color="C0", alpha=0.25, label=r"$\pm 1\sigma$ (XS+flux+reint+DetVar+stat)")
plt.xlabel("reco neutrino energy [MeV]")
plt.ylabel(f"events / bin @ run-{RUN_PERIOD} open-data POT")
plt.title("Recreated fullosc prediction with systematic error band")
plt.legend(); plt.xlim(BINS[0], BINS[-1]); plt.ylim(bottom=0); plt.tight_layout(); plt.show()

## 8. Fractional uncertainty breakdown by source

Reuses `make_sys_frac_error_plot` to show the fractional uncertainty vs reco energy, split into
Total GENIE (XS), Flux, Reinteraction, Total DetVar, and MC stat.

In [ ]:
make_sys_frac_error_plot(
    tot_sys_frac_cov=tot_frac_with_stat,
    tot_pred_sys_frac_cov=tot_frac_with_stat,
    rw_sys_frac_cov_dic=rw_sys_frac_cov_dic,
    detvar_sys_frac_cov_dic=detvar_sys_frac_cov_dic,
    pred_stat_cov=pred_stat_cov,
    data_stat_cov=np.zeros((nb, nb)),
    mc_pred_counts=pred_counts,
    pred_counts=pred_counts,
    display_var="reco neutrino energy [MeV]",
    display_bins=BINS,
    log_x=False,
    include_total=True, include_pred_stat=True, include_data_stat=False,
    include_rw=True, include_detvar=True,
    detvar_df=detvar_df,     # only needs to be non-None
    show=True,
)

## 9. (Optional) Compare with the official `numu2nue` reference file

Download the reference file (run-3 example on pnfs:
`/pnfs/uboone/persistent/users/jjo/processed_chekcout_rootfiles/bnb/run3/cv/checkout_prodgenie_bnb_overlay_numu2nue_ccactive_run3_PF.root`)
into `data_files_location`, then run this cell. It reads the reference truth/reco Enu directly
with uproot and overlays it on the recreated prediction (shape comparison; POT-normalized to
the recreated sample's sum of weights).

In [ ]:
import uproot
REFERENCE_FILE = os.path.join(
    data_files_location, "checkout_prodgenie_bnb_overlay_numu2nue_ccactive_run3_PF.root")

if not os.path.exists(REFERENCE_FILE):
    print("Reference file not downloaded yet -> skipping. Expected at:")
    print(" ", REFERENCE_FILE)
else:
    rf = uproot.open(REFERENCE_FILE)
    ev = rf["wcpselection/T_eval"].arrays(
        ["truth_nuPdg", "truth_isCC", "truth_vtxInside", "weight_cv", "weight_spline"], library="np")
    kine = rf["wcpselection/T_KINEvars"].arrays(["kine_reco_Enu"], library="np")
    m = (np.abs(ev["truth_nuPdg"]) == 12) & ev["truth_isCC"].astype(bool)
    if REQUIRE_FV:
        m &= ev["truth_vtxInside"].astype(bool)
    ref_reco = kine["kine_reco_Enu"][m]
    ref_w = (ev["weight_cv"] * ev["weight_spline"])[m]
    keep = ref_reco > 0
    ref_reco, ref_w = ref_reco[keep], ref_w[keep]
    # normalize reference to the recreated sample's total weighted events (shape comparison)
    ref_w = ref_w * (pred_counts.sum() / np.sum(np.histogram(ref_reco, weights=ref_w, bins=BINS)[0]))

    plt.figure(figsize=(8, 5))
    plt.hist(BIN_CENTERS, bins=BINS, weights=pred_counts, histtype="step", lw=2, label="recreated fullosc")
    plt.bar(BIN_CENTERS, 2 * band, width=np.diff(BINS), bottom=pred_counts - band, alpha=0.25, label=r"recreated $\pm1\sigma$")
    plt.hist(ref_reco, bins=BINS, weights=ref_w, histtype="step", lw=2, label="official numu2nue (shape-normalized)")
    plt.xlabel("reco neutrino energy [MeV]"); plt.ylabel("events / bin (normalized)")
    plt.title("Recreated vs official numu2nue"); plt.legend(); plt.xlim(BINS[0], BINS[-1])
    plt.ylim(bottom=0); plt.tight_layout(); plt.show()

## Caveats and next steps

- **XS ratio is a placeholder** (Section 2) -- replace with a real GENIE total-cross-section
  spline ratio `sigma_nueCC(E)/sigma_numu_total(E)` on Ar40.
- **DetVar is borrowed from nu_overlay.** There is no nue DetVar sample, so the DetVar
  fractional uncertainty vs reco energy is computed from the run-4d `nu_overlay` detector
  variations and applied to this nueCC-appearance prediction. This assumes detector-response
  variations are similar for numu and nue final states.
- **Run mismatch**: recreation uses run 4d; the example official reference is run 3.
- **Reinteraction** is taken from the nue template (nueCC hadronic final state); **flux** from
  the numu donor. Adjust in Section 6 if you prefer a different split.
- Set `USE_DETVAR_BOOTSTRAPPING=True` for the analysis-standard statistical treatment of the
  CV-vs-variation difference (slower). Covariances can be cached via
  `get_rw_sys_frac_cov_matrices` / `get_detvar_sys_frac_cov_matrices` if desired.